In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
ENTREPOT_PATH = "/home/administrateur/Bureau/Datagrosyst/data_entrepot_outils/"
donnees = {}

def import_df(df_name, path_data, sep, index_col=None):
    donnees[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False).replace({'\r\n': '\n'}, regex=True)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

tables_with_id = [
    'composant_culture',
    'culture',
    'espece',
    'recolte_rendement_prix',
    'recolte_rendement_prix_restructure'
]

tables_without_id = [
]


# import des données de l'entrepôt avec la colonne 'id' en index 
import_dfs(tables_with_id, ENTREPOT_PATH, sep = ',', index_col='id', verbose=False)

# import des données du magasin
import_dfs(tables_without_id, ENTREPOT_PATH, sep = ',', verbose=False)

100%|██████████| 5/5 [00:07<00:00,  1.50s/it]
0it [00:00, ?it/s]


In [3]:
culture = donnees['culture'].reset_index().rename(columns={'id':'culture_id'}).copy()
cc = donnees['composant_culture'].reset_index().rename(columns={'id':'composant_culture_id'}).copy()
# espece = donnees['espece'].rename(columns={'id':'espece_id'}).reset_index().copy()
espece = pd.read_csv('/home/administrateur/Bureau/' + 'ref_espece_gcpe.csv').rename(columns={'id':'espece_id'})
with open('/home/administrateur/Bureau/' + 'liste_culture_id_gcpe.txt', "r") as f:
    list_cid_gcpe = [line.strip() for line in f.readlines()]

In [4]:
df = cc.merge(espece, how='left', on='espece_id')
# On ne garde que les culture_id qui ont été utilisées dans un sdc GCPE
culture = culture.loc[culture['culture_id'].isin(list_cid_gcpe)]
df = df.merge(culture, how='right', on='culture_id')

In [5]:
def concat_unique_sorted(series):
    cleaned = series.dropna().unique()
    if len(cleaned) == 0:
        return np.nan
    return '_'.join(sorted(cleaned))
def get_nb_unique_typo(series):
    cleaned = series.dropna().unique()
    return len(cleaned)

In [6]:
agg_dict = {
    'nb_composant_culture': 'sum',
    'typodirodur_espece_precise': concat_unique_sorted,
    'typodirodur_espece_famille_bota': concat_unique_sorted,
    'nb_typodirodur_espece_precise': get_nb_unique_typo,
    'nb_typodirodur_espece_famille_bota': get_nb_unique_typo,
}

In [7]:
len(df.loc[df['composant_culture_id'].isna(), 'nom'].unique().tolist())

193

In [8]:
df['nb_composant_culture'] = 1
df['nb_typodirodur_espece_precise'] = df['typodirodur_espece_precise']
df['nb_typodirodur_espece_famille_bota'] = df['typodirodur_espece_famille_bota']

df2 = df[['culture_id',
         'nb_composant_culture',
         'typodirodur_espece_precise',
         'typodirodur_espece_famille_bota',
         'nb_typodirodur_espece_precise',
         'nb_typodirodur_espece_famille_bota']].groupby('culture_id').agg(agg_dict).reset_index()

In [9]:
len(df2.typodirodur_espece_precise.unique())

1996

In [10]:
df2.typodirodur_espece_precise.value_counts(dropna=False).sort_values(ascending=False)

typodirodur_espece_precise
Blé tendre Hiver Meunier                                           8863
Maïs                                                               8447
Blé tendre Hiver Biscuit                                           6555
Colza Oléagineux Hiver                                             6419
Orge 6 rangs Hiver                                                 4477
                                                                   ... 
Ray-grass d'Italie_Trèfle blanc_Trèfle de Micheli_Trèfle violet       1
Avoine Noir(e) Hiver_Espèces diverses                                 1
Colza Oléagineux Hiver_Fèverole Hiver_Lentille_Lin Oléagineux         1
Avoine Noir(e) Printemps_Pois Fourrager / Fourrage Hiver              1
Mélange Fourrager_Pois Protéagineux Hiver_Vesce commune Hiver         1
Name: count, Length: 1996, dtype: int64

In [11]:
len(df2.typodirodur_espece_famille_bota.unique())

94

In [12]:
df2.typodirodur_espece_famille_bota.value_counts(dropna=False).sort_values(ascending=False)

typodirodur_espece_famille_bota
Poaceae                                        46809
Fabaceae                                       11027
Brassicaceae                                    6870
Fabaceae_Poaceae                                6427
NaN                                             5127
                                               ...  
Convolvulaceae                                     1
Asteraceae_Fabaceae_Poaceae_Rubiaceae              1
Boraginaceae                                       1
Renonculaceae                                      1
Asphodelaceae_Asteraceae_Poaceae_Salicaceae        1
Name: count, Length: 94, dtype: int64

In [13]:
df3 = df2.loc[df2['nb_typodirodur_espece_precise'] != 1]
df3 = df3.typodirodur_espece_precise.value_counts().sort_values(ascending=False).reset_index()
df3.rename(columns={'count':'culture_dirodur'}, inplace=True)
df3.to_csv('/home/administrateur/Bureau/' + "matrice_passage_dirodur_espece_precise.csv", index=False)
df3

,typodirodur_espece_precise,culture_dirodur
0,Pois Fourrager / Fourrage Hiver_Triticale,267
1,Fétuque élevée_Ray-grass anglais_Trèfle blanc,242
2,Blé tendre Hiver Amidon_Blé tendre Hiver Meunier,202
3,Blé tendre Hiver Meunier_Fèverole Hiver,189
4,Ray-grass d'Italie_Trèfle incarnat,178
...,...,...
1735,Avoine Noir(e) Hiver_Pois Fourrager / Fourrage...,1
1736,Fèverole Printemps Grain(es)_Maïs amidon,1
1737,Dactyle_Fétuque des prés_Luzerne_Ray-grass d'I...,1
1738,Blé tendre Printemps Meunier_Cameline,1


In [14]:
df4 = df2.loc[df2['nb_typodirodur_espece_famille_bota'] != 1]
df4 = df4.typodirodur_espece_famille_bota.value_counts().sort_values(ascending=False).reset_index()
df4.rename(columns={'count':'culture_dirodur_famille_bota'}, inplace=True)
df4.to_csv('/home/administrateur/Bureau/' + "matrice_passage_dirodur_famille_bota.csv", index=False)
df4

,typodirodur_espece_famille_bota,culture_dirodur_famille_bota
0,Fabaceae_Poaceae,6427
1,Brassicaceae_Fabaceae,423
2,Brassicaceae_Poaceae,50
3,Asteraceae_Fabaceae,41
4,Brassicaceae_Fabaceae_Polygonaceae,40
5,Asteraceae_Fabaceae_Poaceae,39
6,Brassicaceae_Fabaceae_Poaceae,37
7,Asteraceae_Poaceae,25
8,Fabaceae_Plantaginaceae_Poaceae,20
9,Brassicaceae_Polygonaceae,17


In [15]:
(len(df2)-len(df3))/len(df2)

0.9800593635040512

In [16]:
(len(df2)-len(df4))/len(df2)

0.9993123918449672